# pii

> whether a document is somebody's business, decided by arithmetic rather than by a model

In [ ]:
#| default_exp pii

A vault fills up with things that are nobody else's business: a bank statement, a medical
letter, an exported chat, a CV somebody sent you. Answering a question out of those is exactly
what the vault is for. Sending them to a hosted model is exactly what it is not.

So something has to decide, and that something cannot be a model — a classifier that has to
read the document in order to say whether the document may be read has already lost. It is
arithmetic here: patterns with checksums where a checksum exists, counted, with the count
weighed against the length of what was scanned.

Being *approximately* right is the wrong target. A miss sends somebody's medical history to a
cloud API, and a false positive costs a slower answer from a smaller model. Those are not
symmetric, and nothing here pretends they are.

In [ ]:
#| export
import re
from fastcore.all import AttrDict, L

## What counts

`MAX_SCAN` is Bytes of a document worth scanning. PII is not evenly distributed -- a statement's account
number is in its header -- but a scan is linear and a vault holds whole books, so a long
document is sampled at both ends rather than read entire. `DENSE` Matches per thousand characters above which a document is *about* people rather than merely mentioning one. Nothing here uses it to decide `has_pii`; it is for a caller that wants to tell a signature block apart from a customer list.

In [ ]:
#| export
MAX_SCAN, DENSE = 200_000, 1.0

In [ ]:
#| export
def luhn(s:str) -> bool:
    "The check digit every payment card carries. Sixteen digits that fail it are not a card."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) < 12: return False
    tot, parity = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == parity: d *= 2; d -= 9 if d > 9 else 0
        tot += d
    return tot % 10 == 0

def _iban_ok(s:str) -> bool:
    "IBAN's mod-97 check: move the country prefix to the end, letters to digits, remainder must be 1."
    s = re.sub(r'[^A-Za-z0-9]', '', s).upper()
    if not (15 <= len(s) <= 34): return False
    t = s[4:] + s[:4]
    try: n = int(''.join(str(int(c, 36)) for c in t))
    except ValueError: return False
    return n % 97 == 1

def _nhs_ok(s:str) -> bool:
    "The UK NHS number's mod-11 check digit. Ten digits in a row are otherwise just ten digits."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 10: return False
    tot = sum(d * (10 - i) for i, d in enumerate(ds[:9]))
    chk = 11 - tot % 11
    return chk != 10 and (0 if chk == 11 else chk) == ds[9]

def _ssn_ok(s:str) -> bool:
    "A US SSN's structurally impossible cases, which is as much as arithmetic can say about one."
    ds = re.sub(r'\D', '', s)
    if len(ds) != 9: return False
    a, b, c = ds[:3], ds[3:5], ds[5:]
    return a not in ('000', '666') and a[0] != '9' and b != '00' and c != '0000'

A checksum is the difference between a detector and a superstition. Sixteen digits in a
document are usually an order number; sixteen digits that pass Luhn are a card. Where a
standard gives us one — cards, IBANs, NHS numbers — it is checked, and where it does not
(email, phone) the pattern carries the whole weight and is written tightly enough to.

In [ ]:
#| export
#: kind -> (pattern, validator or None). Order matters only for reporting; spans are
#: de-overlapped afterwards, longest first, so a card inside a longer digit run wins.
PATTERNS = {
    'email':   (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', None),
    'card':    (r'\b(?:\d[ -]?){13,19}\b', luhn),
    'iban':    (r'\b[A-Z]{2}\d{2}[ ]?(?:[A-Z0-9]{4}[ ]?){2,7}[A-Z0-9]{1,4}\b', _iban_ok),
    'ssn':     (r'\b\d{3}-\d{2}-\d{4}\b', _ssn_ok),
    'nhs':     (r'\b\d{3}[ -]?\d{3}[ -]?\d{4}\b', _nhs_ok),
    'phone':   (r'\+\d{1,3}[ .-]?\(?\d{1,5}\)?[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\(\d{2,5}\)[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b0\d{1,4}[ .-]\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b\d{3}-\d{3}-\d{4}\b'
                r'|\b(?:phone|tel|telephone|mobile|cell|fax)\b\W{0,8}\+?[\d ().-]{7,20}\d', None),
    'ip':      (r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', None),
    'dob':     (r'\b(?:date of birth|dob|born)\b\W{0,12}(?:\d{1,4}[/-]\d{1,2}[/-]\d{1,4}|\d{1,2} \w+ \d{4})', None),
    'passport':(r'\b(?:passport(?:\s*(?:no|number|#))?)\W{0,6}[A-Z0-9]{6,9}\b', None),
    'licence': (r'\b(?:driver.?s? licen[cs]e|dl)(?:\s*(?:no|number|#))?\W{0,6}[A-Z0-9]{5,20}\b', None),
    'account': (r'\b(?:account|acct|a/c)(?:\s*(?:no|number|#))?\W{0,6}\d{6,17}\b', None),
    'sortcode':(r'\b(?:sort\s*code)\W{0,6}\d{2}[- ]?\d{2}[- ]?\d{2}\b', None),
    'secret':  (r'\b(?:sk-[A-Za-z0-9_-]{16,}|ghp_[A-Za-z0-9]{20,}|xox[baprs]-[A-Za-z0-9-]{10,}|AKIA[0-9A-Z]{16}|AIza[0-9A-Za-z_-]{35})\b', None),
    'medical': (r'\b(?:diagnos\w+|prescri\w+|patient (?:id|number|name)|nhs number|medical record)\b', None),
}
IDENTIFYING = frozenset({'email', 'card', 'iban', 'ssn', 'nhs', 'phone', 'dob', 'passport', 'licence', 'account',
                         'sortcode', 'medical'})
_COMPILED = {k: (re.compile(p, re.I), v) for k, (p, v) in PATTERNS.items()}

## Where it is

In [ ]:
#| export
def _scan_text(text:str, mx:int=MAX_SCAN) -> str:
    "The part of a long document worth scanning: both ends, since headers and footers carry the identity."
    text = str(text or '')
    if len(text) <= mx: return text
    half = mx // 2
    return text[:half] + '\n' + text[-half:]

def pii_spans(text:str,          # what to scan
              kinds=None,        # restrict to these kinds; None -> every pattern
              mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
) -> L:
    """Every match, as `(start, end, kind, text)`, longest first and never overlapping.

    Overlap is not a detail. A sixteen-digit card matches `card`, `phone` and `nhs`, and
    counting it three times turns one payment card into a document that is apparently dense
    with identity. The longest match wins and the rest are dropped.
    """
    text = _scan_text(text, mx)
    want = set(kinds or _COMPILED)
    found = []
    for kind, (rx, ok) in _COMPILED.items():
        if kind not in want: continue
        for m in rx.finditer(text):
            if ok is not None and not ok(m.group(0)): continue
            found.append((m.start(), m.end(), kind, m.group(0)))
    found.sort(key=lambda s: (s[0] - s[1], s[0]))       # longest first, then leftmost
    out, taken = [], []
    for s, e, kind, val in found:
        if any(s < te and ts < e for ts, te in taken): continue
        taken.append((s, e))
        out.append((s, e, kind, val))
    return L(sorted(out))

In [ ]:
#| export
def pii_report(text:str,          # what to scan
               kinds=None,        # restrict to these kinds; None -> every pattern
               mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
) -> AttrDict:
    """What was found and whether it makes this text somebody's business.

    `has_pii` is what a policy switches on, and it is deliberately narrow: an IP address or an
    API key is reported but does not on its own make a document private, because a server log
    is not somebody's private life and treating it as one costs every question about
    infrastructure a smaller model for nothing.
    """
    spans, counts = pii_spans(text, kinds, mx), {}
    for _, _, k, _ in spans: counts[k] = counts.get(k, 0) + 1
    n = len(_scan_text(text, mx))
    ident = {k: v for k, v in counts.items() if k in IDENTIFYING}
    return AttrDict(has_pii=bool(ident), kinds=counts, identifying=ident, n=len(spans),
                    scanned=n, density=round(1000 * len(spans) / max(n, 1), 3), spans=spans)

`has_pii` is the switch, and it is narrower than `n > 0` on purpose. A document with an IP
address in it is a log; a document with an email address in it is about a person. Only the
second should cost the answer a smaller model, and conflating them means every question about
a server ends up on the slow path for nothing.

In [ ]:
#| export
def redact(text:str,       # the text to mask
           spans=None,     # spans from `pii_spans`; recomputed over the whole of `text` when None
           kinds=None,     # restrict to these kinds
           mask:str=None,  # what to put in place of a match; None -> `[KIND]`
) -> str:
    """`text` with every match replaced, so what is left can be read by anything.

    Applied right to left, because replacing left to right moves every span after the one just
    replaced and the offsets stop meaning anything two matches in.

    And scanned entire, however long it is. `pii_spans` samples a long document at both ends,
    which is the right trade when the question is *whether* there is anything here; it is the
    wrong one when the answer is going to be used as offsets, because an offset into a sample
    addresses the wrong characters of the text it came from -- masking the middle of the
    document and leaving the end of it in the clear.
    """
    out = str(text or '')
    if spans is None: spans = pii_spans(out, kinds, mx=len(out))
    for s, e, kind, _ in sorted(spans, reverse=True):
        out = out[:s] + (mask if mask is not None else f'[{kind.upper()}]') + out[e:]
    return out

## Asking the vault

`Vault.pii` is the document-level question — "is this thing private" — and `pii_ctx` is the
one that actually gates an answer, because `ask` does not send a document, it sends whichever
sections retrieval chose. A vault can hold one bank statement among four hundred papers and
the question decides whether it is in the room.

In [ ]:
#| export
from vishalakshi.core import Vault
from fastcore.all import patch

@patch
def pii(self:Vault,
        ref,                 # a doc_id, source, title or path -- whatever `document` takes
        max_chars:int=MAX_SCAN,
) -> AttrDict:
    "Whether one whole document is somebody's business, and what in it says so."
    d = self.document(ref, max_chars=max_chars)
    r = pii_report(d.text)
    override = (d.get('meta') or {}).get('pii_override')
    r.detected, r.override = r.has_pii, override
    if override == 'clear': r.has_pii = False
    r.doc_id, r.title, r.source = d.get('doc_id'), d.get('title'), d.get('source')
    return r

@patch
def mark_not_pii(self:Vault, ref, clear:bool=True, reason:str='') -> dict:
    'Explicitly clear a false-positive PII decision, or restore automatic detection.'
    d = self.doc(ref)
    if not d: raise ValueError(f'no document in the vault matching {ref!r}')
    return self.set_meta(d['id'], pii_override='clear' if clear else None, pii_override_reason=reason if clear else '')

def pii_ctx(ctx) -> AttrDict:
    """The report for an assembled context -- which is what a policy has to gate on.

    `ask` never sends a document. It sends the sections retrieval chose, so a vault holding one
    private letter among four hundred papers is only a private question when the letter is
    among the sections, and asking the document is the wrong question at the wrong time.
    """
    parts = [str(getattr(r, 'text', None) or (r.get('text') if isinstance(r, dict) else '') or '')
             for r in (list(ctx.get('results') or []) + list(ctx.get('related') or []))]
    return pii_report('\n\n'.join(parts))

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Using it

In [ ]:
report = pii_report("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00, account number 12345678.
Server 10.0.0.14 returned 500. Order number 4471000012345678.
""")
report.has_pii, report.identifying, report.n

(True, {'email': 1, 'phone': 1, 'card': 1, 'sortcode': 1, 'account': 1}, 6)

In [ ]:
from fastcore.test import test_eq
test_eq(report.has_pii, True)
test_eq('card' in report.identifying, True)      # passes Luhn
test_eq(report.kinds.get('ip'), 1)               # reported...
test_eq('ip' in report.identifying, False)       # ...but a log is not somebody's private life

In [ ]:
print(redact("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00.
""").strip())

Invoice 4471 for Ada Lovelace <[EMAIL]>, [PHONE].
Card [CARD], [SORTCODE].


"Ada Lovelace" is still there, and that is the boundary rather than an oversight. Every pattern
above is one a machine can *verify* — a checksum, or a shape tight enough to stand in for one.
Which words in a document are a person's name is the judgement this module exists to avoid, so
`redact` masks what can be recognised and does not pretend to anonymise. It is enough to keep an
account number off the network and not enough to make a letter unattributable; `ask(pii='local')`
is the setting for the second, because it sends nothing at all.

A number that fails its checksum is not the thing the checksum protects. This is the whole
argument for doing it arithmetically rather than by asking a model whether something "looks
like" a card number.

In [ ]:
test_eq(pii_report('Order 4111 1111 1111 1112 shipped').has_pii, False)   # fails Luhn
test_eq(pii_report('Card 4111 1111 1111 1111 charged').has_pii, True)     # passes it
test_eq(pii_report('The build takes 20 minutes and costs nothing.').has_pii, False)

In [ ]:
#| hide
long_doc = 'Account number 12345678\n' + ('filler text. ' * 40_000) + '\nsigned, ada@example.com'
r = pii_report(long_doc)
test_eq(r.has_pii, True)
test_eq(sorted(r.identifying), ['account', 'email'])
test_eq(r.scanned <= MAX_SCAN + 1, True)

# ...but a *report* may sample and a redaction may not. Masking the same document by the offsets
# a sampled scan produced left the tail address in the clear, which is the one thing `redact`
# exists to prevent.
masked = redact(long_doc)
assert 'ada@example.com' not in masked, masked[-80:]
assert '12345678' not in masked, masked[:80]
test_eq(masked.count('filler text. '), 40_000)      # and nothing in between was moved
test_eq(pii_report(masked).has_pii, False)

In [ ]:
#| hide
one = pii_report('4111 1111 1111 1111')
test_eq(one.n, 1)
test_eq(list(one.kinds), ['card'])

A run of digits is not a phone number, and this is where a loose detector does its damage:
every order reference in the vault becomes somebody's landline, every document becomes private,
and every question about them gets the slow path. The table is the argument for the shape of
the patterns.

In [ ]:
#| hide
# What must and must not put a document on the local-only path. A miss here sends somebody's
# medical history to a hosted API; a false positive costs a slower answer. Both belong in a
# test, and they are not the same size of mistake.
cases = [
    ('Order 4111 1111 1111 1112 shipped',                 False),   # fails Luhn
    ('Card 4111 1111 1111 1111 charged',                  True),
    ('phone 020 7946 0958',                               True),
    ('+44 20 7946 0958',                                  True),
    ('call (555) 123-4567',                               True),
    ('555-123-4567',                                      True),
    ('ada@example.com',                                   True),
    ('The build takes 20 minutes and costs nothing.',     False),
    ('Run 2024 1000 2000 3000 through the pipeline',      False),
    ('Release 1.2.3 shipped on 2024-05-01 with 400 tests', False),
    ('Server 10.0.0.14 returned 500',                     False),   # reported, not identifying
    ('commit 8f3a2b1 touched 120 lines in 14 files',      False),
]
for text, want in cases: test_eq((text, pii_report(text).has_pii), (text, want))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()